# カワウソ黒板解説 ／ 動画レンダリング

台本を書き換えて上から実行すると **MP4 が出ます。**

**先にアップロードするもの**（左のファイルペインにドラッグ）

| ファイル | 要否 | 備考 |
| --- | --- | --- |
| `render.py` `mkpng.py` `sound.py` `cutout.py` | 必須 | |
| キャラ画像5枚 | 必須 | 名前は問わない。手順2で対応づける。**透過してなくても抜きます** |
| `narration.wav` | 推奨 | 無ければ無音で書き出します |
| `bgm.mp3` | 任意 | 本編用 |
| `bgm_climax.mp3` | 任意 | オチ用。無ければ本編用を流用 |

**字幕ファイルは要りません。** テロップは映像に直接描いているので、SRTの読み込みも
自動字幕の修正も不要です。

**やらないこと**
- キャラ画像の生成（一貫性が落ちるので外で作る）
- 投稿の自動化（**1日1本、手で出す**。YouTubeの量産判定を避けるため）


## 1. 準備

In [ ]:
!apt-get -qq install -y fonts-ipafont-gothic > /dev/null
!pip -q install pillow numpy
!ffmpeg -version | head -1
import os; os.makedirs('assets', exist_ok=True)
print('OK')

## 2. キャラ画像を透過して assets/ に入れる

**アップロードしたファイル名を左に、割り当てたい役を右に**書いてください。
チェック柄が焼き込まれていても、白背景でも、ここで抜きます。

| 役 | どの絵か |
| --- | --- |
| `char_explain.png` | 笑顔・片手を上げている（**一番多く出る**） |
| `char_surprise.png` | 驚き・両手を上げている |
| `char_serious.png` | 真面目／怒り顔 |
| `char_proud.png` | 目を閉じた微笑み・手は下 |
| `char_blink.png` | 目を閉じている（**解説顔と同じポーズのもの**） |

In [ ]:
MAP = {
    'アップロードした名前1.png': 'char_explain.png',
    'アップロードした名前2.png': 'char_surprise.png',
    'アップロードした名前3.png': 'char_serious.png',
    'アップロードした名前4.png': 'char_proud.png',
    'アップロードした名前5.png': 'char_blink.png',
}
args = ' '.join('"%s:%s"' % (k, v) for k, v in MAP.items() if os.path.exists(k))
if args:
    !python3 cutout.py --name --tol 34 {args}
else:
    print('※ ファイルが見つかりません。MAP の左側を実際のファイル名に直してください。')
    print('   このまま進めるとグレーの代役で書き出されます（流れの確認用）。')

## 3. 黒板の図をつくる

水分子・衝突・氷の格子を透過PNGで書き出します。新しい図が要るときは `mkpng.py` に足す。

In [ ]:
!python3 mkpng.py
from IPython.display import Image as IPImage, display
display(IPImage('assets/01_water_molecule.png', width=160))

## 4. 台本

**毎回ここだけ書き換えます。**

1ショット = 画面が変わる1単位。**1.0〜2.2秒**に収める（参考動画の実測が1.3〜1.7秒）。

| キー | 意味 |
| --- | --- |
| `dur` | このショットの長さ（秒） |
| `kind` | `wide` 黒板＋キャラ ／ `board` 黒板に寄る ／ `face` 顔アップ ／ `title` 黒背景 |
| `face` | `surprise` `explain` `serious` `proud` |
| `board` | 黒板に出す行。`dict(text=..., size=..., color=MUSTARD)` も可 |
| `tele` | 画面下のテロップ。`{ }` で囲んだ語だけ黄色 |
| `fig` | `(ファイル名, 黒板幅に対する比, 何秒で1回転するか)` |
| `tap` | `True` で指し棒が黒板を叩く |

**同じ `kind` を3回続けない。**`wide` が続いたら `face` か `board` を挟む。

In [ ]:
import importlib, render
importlib.reload(render)
from render import MUSTARD, CRIMSON

render.SHOTS = [
 dict(dur=1.5, kind='wide',  face='surprise', board=['電子レンジは','{何を温めてる？}'],
      tele='電子レンジって、'),
 dict(dur=1.1, kind='face',  face='surprise', tele='{何を温めてる}と思う？'),
 dict(dur=0.8, kind='title', title='ほぼ全員\n間違えます'),

 dict(dur=1.6, kind='wide',  face='explain', board=['正解は'], tele='正解は、'),
 dict(dur=1.5, kind='board', board=['{食べ物じゃない}'], tele='{食べ物じゃありません。}'),
 dict(dur=1.4, kind='face',  face='explain', tele='中に入っている、'),
 dict(dur=1.5, kind='board', board=[dict(text='水', size=300, color=MUSTARD)],
      tele='{水}です。', tap=True),

 dict(dur=1.6, kind='wide',  face='serious', board=['電子レンジが','出しているのは'],
      tele='電子レンジが出しているのは、'),
 dict(dur=1.8, kind='board', board=[dict(text='マイクロ波', size=140, color=MUSTARD)],
      tele='{マイクロ波}という、'),
 dict(dur=1.5, kind='wide',  face='serious', board=['目に見えない','電波'],
      tele='目に見えない電波。'),
 dict(dur=2.0, kind='board', fig=('01_water_molecule.png', .58, 1.1),
      tele='これを浴びると、水の分子が'),
 dict(dur=1.9, kind='board', fig=('01_water_molecule.png', .40, 0.45),
      board=[dict(text='ぐるぐる回る', size=110)], tele='{猛烈に}回りはじめます。'),
 dict(dur=1.8, kind='wide',  face='serious',
      board=[dict(text='1秒に\n24億回', size=150, color=MUSTARD)],
      tele='一秒間に、{24億回}。', tap=True),

 dict(dur=1.4, kind='face',  face='explain', tele='向きを変えるたびに、'),
 dict(dur=2.0, kind='board', fig=('02_collision.png', .92, None),
      tele='隣の分子と{ぶつかり合う。}'),
 dict(dur=1.6, kind='wide',  face='explain',
      board=[dict(text='ぶつかる → 熱', size=96, color=CRIMSON)],
      tele='そこで{熱}が生まれて、', tap=True),
 dict(dur=1.6, kind='board', board=['食べ物を','{内側から}温める'],
      tele='食べ物を内側から温めている。'),
 dict(dur=1.6, kind='face',  face='explain', tele='つまり、外から温めてない。'),

 dict(dur=0.9, kind='title', title='ここで\n1つ気づく'),
 dict(dur=1.7, kind='wide',  face='explain', board=['じゃあ','{お皿}は？'],
      tele='じゃあ、{お皿}はどうなる？'),
 dict(dur=1.6, kind='board', board=[dict(text='温まらない', size=150, color=MUSTARD)],
      tele='水分がないので、{温まりません。}', tap=True),
 dict(dur=1.7, kind='face',  face='explain', tele='お皿が熱いのは、'),
 dict(dur=1.7, kind='wide',  face='explain', board=['食べ物から','熱が移っただけ'],
      tele='食べ物から{熱が移っただけ}なんです。'),

 dict(dur=1.3, kind='title', title='つまり'),
 dict(dur=1.7, kind='wide',  face='proud', board=['{温めてない}'],
      tele='電子レンジは、温めているんじゃない。'),
 dict(dur=2.2, kind='board', board=[dict(text='水を\n暴れさせてる', size=160, color=MUSTARD)],
      tele='{水を暴れさせている}だけなんです。', tap=True),
 dict(dur=1.5, kind='face',  face='proud', tele=''),

 dict(dur=2.4, kind='wide',  face='proud', board=['次は','{飛行機}'],
      tele='次は、飛行機がなんで飛ぶのか。'),
 dict(dur=2.6, kind='board', board=[dict(text='学校の説明は\n間違い', size=120, color=CRIMSON)],
      tele='学校で習ったあの説明、{実は間違いです。}'),
]

render.DURATION = render.build()
from collections import Counter
print('尺 %.1f秒 / %dショット / 平均 %.2f秒' %
      (render.DURATION, len(render.SHOTS), render.DURATION/len(render.SHOTS)))
print(Counter(s['kind'] for s in render.SHOTS))
long = [(i+1, s['dur']) for i, s in enumerate(render.SHOTS) if s['dur'] > 2.2]
print('※ 2.2秒を超えるショット:', long if long else 'なし')

## 5. 全ショットを一覧で確認する

書き出す前に、ここで**絵の流れ**を見ます。同じ絵が続いていないか、
テロップがキャラに被っていないかを確認して、おかしければ手順4に戻る。

In [ ]:
# 台本を編集した状態のまま描くので、ここで reload してはいけない
import math
from PIL import Image
from IPython.display import display

cols = 6; rows = math.ceil(len(render.SHOTS)/cols); tw, th = 150, 267
sheet = Image.new('RGB', (cols*tw, rows*th), (24,26,24))
for i, s in enumerate(render.SHOTS):
    im = render.render_frame(s['t'] + s['dur']*0.6).resize((tw-3, th-3), Image.LANCZOS)
    sheet.paste(im, ((i%cols)*tw+2, (i//cols)*th+2))
display(sheet)

## 6. 効果音とBGMの設計

`sound.py` が **効果音そのものを合成** して `se.wav` を作ります。素材を探す必要はありません。

- カットの切り替わり → 「スッ」
- 章の切り替わり（`title`）→ 「ドン」
- 指し棒のタップ → 「トン」

**BGMは無音の区間を2つ作ります。**冒頭のフックと、オチの直前。
とくに**オチ直前の無音が一番効きます。**

In [ ]:
# 別プロセスで走らせると手順4の編集が反映されないので、ここで直接呼ぶ
import importlib, sound
importlib.reload(sound)                 # sound は render を参照するだけなので台本は消えない

ev = sound.se_events()
sound.write_wav('se.wav', sound.build_track(render.DURATION))

from collections import Counter
c = Counter(k for _, k in ev)
print('効果音 %d個 / %.1f秒 = 平均 %.2f秒に1回' % (len(ev), render.DURATION, render.DURATION/len(ev)))
print('  スッ（カット切替） %d 個' % c['whoosh'])
print('  ドン（章の切替）   %d 個' % c['don'])
print('  トン（棒で叩く）   %d 個' % c['ton'])

print('\nBGMの音量設計')
for s, e, v, why in sound.BGM_PLAN:
    print('  %5.1f〜%5.1f秒  %3d%%   %s' % (s, min(e, render.DURATION), v*100, why))

import numpy as np
demo = np.concatenate([sound.whoosh(), np.zeros(sound.SR//3), sound.don(),
                       np.zeros(sound.SR//3), sound.ton()])
sound.write_wav('se_preview.wav', demo)
from IPython.display import Audio, display
print('\nスッ / ドン / トン:'); display(Audio('se_preview.wav'))

## 7. ナレーション音声

**A案（おすすめ）** CapCutかVOICEVOXアプリで読み上げを作って `narration.wav` をアップロード。
台本の `tele` をつなげたものが原稿になります（下のセルで書き出せます）。

**B案** 無音のまま進める。あとから足せます。

In [ ]:
txt = []
for i, s in enumerate(render.SHOTS):
    line = s.get('tele','').replace('{','').replace('}','')
    if line: txt.append('%5.1fs  %s' % (s['t'], line))
open('narration_script.txt','w').write('\n'.join(txt))
print('\n'.join(txt))
print('\n-> narration_script.txt に保存しました')
import os
print('\nnarration.wav:', '有り' if os.path.exists('narration.wav') else '無し（無音で書き出します）')

## 8. 全フレーム書き出し

47秒 × 30fps ≒ 1400枚。数分かかります。

In [ ]:
import os, time
os.makedirs('frames', exist_ok=True)
n = int(render.DURATION * render.FPS)
t0 = time.time()
for i in range(n):
    render.render_frame(i/render.FPS).save('frames/%05d.png' % i)
    if i % 150 == 0: print('  %d / %d  (%.0fs)' % (i, n, time.time()-t0), flush=True)
print('完了 %d枚 / %.0f秒' % (n, time.time()-t0))

## 9. MP4に合成

映像 ＋ ナレーション ＋ 効果音 ＋ BGM（音量オートメーション付き）をまとめます。
テロップは映像に描き込み済みなので、字幕の処理は要りません。

In [ ]:
import os, subprocess, shlex, sound, importlib
importlib.reload(sound)

FPS = render.FPS
has = lambda f: os.path.exists(f)
cmd = ['ffmpeg','-y','-framerate',str(FPS),'-i','frames/%05d.png']

srcs = []                                    # (ファイル, 種類)
if has('narration.wav'): srcs.append(('narration.wav','nar'))
if has('se.wav'):        srcs.append(('se.wav','se'))
if has('bgm.mp3'):       srcs.append(('bgm.mp3','bgm'))
for f, _ in srcs: cmd += ['-i', f]

if srcs:
    parts, labels = [], []
    for i, (f, kind) in enumerate(srcs, start=1):
        if kind == 'bgm':
            parts.append("[%d:a]volume='%s':eval=frame[b]" % (i, sound.volume_expr()))
            labels.append('[b]')
        else:
            labels.append('[%d:a]' % i)
    mix = ''.join(labels) + 'amix=inputs=%d:duration=first:normalize=0[a]' % len(labels)
    cmd += ['-filter_complex', ';'.join(parts + [mix]) if parts else mix,
            '-map','0:v','-map','[a]']

cmd += ['-c:v','libx264','-preset','medium','-crf','19','-pix_fmt','yuv420p',
        '-r',str(FPS),'-c:a','aac','-b:a','192k','-shortest','ep01.mp4']

print('音声トラック:', [k for _, k in srcs] or 'なし（無音）')
print(' '.join(shlex.quote(c) for c in cmd), '\n')
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stderr[-1500:] if r.returncode else '書き出し成功 -> ep01.mp4')

## 10. 確認してダウンロード

In [ ]:
from IPython.display import HTML, display
import base64, os
print('%.1f MB' % (os.path.getsize('ep01.mp4')/1e6))
b = base64.b64encode(open('ep01.mp4','rb').read()).decode()
display(HTML('<video controls style="max-height:70vh" src="data:video/mp4;base64,%s"></video>' % b))

In [ ]:
from google.colab import files
files.download('ep01.mp4')

---

## 2本目以降

**手順 4 → 5 → 6 → 8 → 9 → 10** だけ。
キャラ画像と図は使い回せるので、書き換えるのは台本のリストだけです。

## 投稿前の確認

- **テロップだけで意味が通るか**（音を切って見る人が7割）
- **2.2秒を超えるショットが無いか**（手順4の最後に出ます）
- **同じ `kind` が3回続いていないか**
- 尺が**35〜50秒**に収まっているか

**投稿は手で。1日1本まで。**
